<a href="https://colab.research.google.com/github/AhmedMahmoud-123/FlyRank_AI/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Paper findings

**Finding A — Growth Prediction (Part IV):** 90% accuracy on new pages from known brands,
75% on entirely unseen brands — a 15-point drop.

**Methodology questions:**
- What defines "known brand" in the split — is a brand's data ever present in both train
  and test (even different pages), and if so, does the model learn brand-level shortcuts
  (a brand's typical publishing cadence, niche, or baseline health score) rather than
  page-level growth signal? This is the same client-leakage question notebook 02 raised
  about `client_hash_id`.
- The report says this was "tested 20 different ways across both methods" with accuracy
  ranging 64%–85% on unseen brands — that's a wide range. What does the *distribution* look
  like, not just the average? A model that's 85% on some brand splits and 64% on others is
  a very different honesty story than a model that's consistently ~75%.
- Top predictor is "Days Visible" (0.16) — is that measured in a window that could overlap
  the label window (a page counted as "visible" during the same days used to judge whether
  it grew)? The paper doesn't show the feature-label timing explicitly.

**Finding B — Zombie Recovery (Part IV):** 99% same-brand vs 97% unseen-brand — only a
2-point gap, much tighter than Finding A's 15-point gap.

**Methodology questions:**
- Why does this model generalize to new brands so much better than Growth Prediction?
  One honest hypothesis: recovery may be driven by broadly transferable signals (content
  age, prior impressions) rather than brand-specific patterns — worth checking whether
  the top features here (Content Age, Impressions) are less brand-coupled than Growth
  Prediction's top feature (Days Visible).
- 59% of zero-traffic pages "came back on their own" — is "recovery" measured over a fixed
  window after the zero-traffic point, and is that window long enough that some "recoveries"
  are actually just noisy sparse-traffic pages crossing the zero threshold randomly rather
  than a real signal-driven comeback?
- The report is written by the company selling the automated fix for exactly this problem
  (zombie recovery). That's not a reason to dismiss the number, but it's a reason to check
  whether the reported metric (accuracy) is the most honest one — precision/recall on the
  minority "does NOT recover" class matters more for a triage tool than raw accuracy, and
  isn't reported here.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# reuse df, X, y, feature_cols from w05_model.ipynb
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# BEFORE: random split (what a naive run would show)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xr_tr, yr_tr)
random_score = rf_random.predict_proba(Xr_te)[:, 1]

# AFTER: grouped split (already have this from Week 5 — recomputed here for the side-by-side)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
Xg_tr, Xg_te = X.iloc[train_idx], X.iloc[test_idx]
yg_tr, yg_te = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(Xg_tr, yg_tr)
grouped_score = rf_grouped.predict_proba(Xg_te)[:, 1]

for label, scores, labels in [('random split', random_score, yr_te), ('grouped split', grouped_score, yg_te)]:
    p20 = precision_at_k(scores, labels.values, 20)
    print(f'{label:14} precision@20: {p20:.3f}   base rate: {labels.mean():.3f}')

NameError: name 'X' is not defined

**Before/after:** [fill after running — state the two precision@20 numbers and the gap.
If grouped is lower, that gap IS the finding: it's how much the random split was letting
the model memorize client-level patterns rather than learning something that generalizes.]

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# 1. label-derived / sibling check — train with vs without each feature that looks suspiciously strong
importances = pd.Series(rf_grouped.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

top_feature = importances.index[0]
X_without = Xg_tr.drop(columns=[top_feature])
rf_check = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_without, yg_tr)
score_without = rf_check.predict_proba(Xg_te.drop(columns=[top_feature]))[:, 1]
print(f'precision@20 WITH {top_feature}:    {precision_at_k(grouped_score, yg_te.values, 20):.3f}')
print(f'precision@20 WITHOUT {top_feature}: {precision_at_k(score_without, yg_te.values, 20):.3f}')

# 2. window overlap check — every feature column name, manually confirmed pre-label-window
label_window_fields = {'impressions_last_30d', 'clicks_last_30d', 'trend_pct', 'trend_direction'}
assert not (set(feature_cols) & label_window_fields), "leakage: a feature is in the label window"
print("no label-window fields in the final feature set — OK")

# 3. product-flag check — confirm no existing-system score/flag used as a feature
product_flags = {'competition_level', 'action_score', 'reason_code'}  # from w03/w04 excluded list
assert not (set(feature_cols) & product_flags), "leakage: a product flag is in the features"
print("no product-decision flags in the features — OK")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.